[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bruskez/market-basket-analysis-imdb/blob/main/market_basket_analysis.ipynb)

# Market-Basket Analysis on the IMDb Top 1000 Dataset

**Goal**: find groups of actors who recur frequently together in the same movies,
treating each movie as a *basket* and the actors in `Star1`-`Star4` as *items*.

**Implemented (from scratch)**: Apriori · PCY · Association rules

**Structure**: (1) data loading · (2) Apriori · (3) PCY · (4) association rules · (5) insights · (6) conclusions

In [ ]:
import itertools
import hashlib
import time
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
%matplotlib inline

## Dataset retrieval

In [ ]:
import os, zipfile

# Kaggle credentials – replace with your own if needed
os.environ['KAGGLE_USERNAME'] = "bruskez"
os.environ['KAGGLE_KEY'] = "KGAT_c5508c04d6abb3f2e6ac2aa84d7f2b73"

!pip install kaggle -q
!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows

with zipfile.ZipFile("imdb-dataset-of-top-1000-movies-and-tv-shows.zip", "r") as z:
    z.extractall(".")

CSV_PATH = "imdb_top_1000.csv"
print("Dataset ready:", CSV_PATH)

## 1. Data loading and basket construction

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Rows in dataset:", len(df))
df[["Series_Title", "Star1", "Star2", "Star3", "Star4"]].head()

In [ ]:
def load_baskets(dataframe):
    # Each movie becomes a set of actors (order-independent, no duplicates)
    star_cols = ["Star1", "Star2", "Star3", "Star4"]
    baskets, titles = [], []
    for _, row in dataframe.iterrows():
        items = {row[c] for c in star_cols if pd.notna(row[c])}
        if items:
            baskets.append(items)
            titles.append(row["Series_Title"])
    return baskets, titles

baskets, titles = load_baskets(df)
N = len(baskets)
print(f"Number of baskets (movies): {N}")
print(f"Number of unique actors:    {len({a for b in baskets for a in b})}")
print(f"Average basket size:        {sum(len(b) for b in baskets)/N:.2f}")

## 2. Apriori algorithm

Exploits the **downward-closure** property: if an itemset is not frequent,
no superset can be. Iterates over increasing sizes `k`:
count singletons (L1) → generate candidates by joining L_{k-1} → prune → count → repeat.

In [ ]:
def apriori_singletons(baskets, min_support):
    counts = defaultdict(int)
    for basket in baskets:
        for item in basket:
            counts[frozenset([item])] += 1
    return {iset: c for iset, c in counts.items() if c >= min_support}


def apriori_generate_candidates(prev_frequent, k):
    """Join step + prune step: every (k-1)-subset of a candidate must be frequent."""
    prev_sets = list(prev_frequent.keys())
    candidates = set()
    n = len(prev_sets)
    for i in range(n):
        for j in range(i + 1, n):
            union = prev_sets[i] | prev_sets[j]
            if len(union) != k:
                continue
            if all(frozenset(sub) in prev_frequent
                   for sub in itertools.combinations(union, k - 1)):
                candidates.add(union)
    return candidates


def apriori_count_candidates(baskets, candidates, k):
    """Count via k-subsets of each basket – O(1) membership lookup on the candidate set."""
    counts = defaultdict(int)
    for basket in baskets:
        if len(basket) < k:
            continue
        for combo in itertools.combinations(sorted(basket), k):
            c = frozenset(combo)
            if c in candidates:
                counts[c] += 1
    return counts


def run_apriori(baskets, min_support, max_k=10, verbose=False):
    levels, stats = {}, {}

    t0 = time.perf_counter()
    L1 = apriori_singletons(baskets, min_support)
    t1 = time.perf_counter()
    levels[1] = L1
    stats[1] = {"n_candidates": len(L1), "n_frequent": len(L1), "time_sec": t1 - t0}

    k, prev = 2, L1
    while prev and k <= max_k:
        t0 = time.perf_counter()
        candidates = apriori_generate_candidates(prev, k)
        if not candidates:
            break
        counts = apriori_count_candidates(baskets, candidates, k)
        Lk = {iset: c for iset, c in counts.items() if c >= min_support}
        t1 = time.perf_counter()
        stats[k] = {"n_candidates": len(candidates), "n_frequent": len(Lk), "time_sec": t1 - t0}
        if verbose:
            print(f"k={k}: {len(candidates)} candidates -> {len(Lk)} frequent ({t1-t0:.4f}s)")
        if not Lk:
            break
        levels[k] = Lk
        prev, k = Lk, k + 1

    return levels, stats

### Run with `min_support = 3`

In [ ]:
MIN_SUPPORT = 3
levels, apriori_stats = run_apriori(baskets, MIN_SUPPORT, verbose=True)

for k in sorted(levels):
    print(f"\nTop frequent itemsets of size {k}:")
    top = sorted(levels[k].items(), key=lambda x: -x[1])[:5]
    for iset, c in top:
        print(f"  {{{', '.join(sorted(iset))}}}  support={c}")

In [ ]:
sizes  = sorted(levels.keys())
counts = [len(levels[k]) for k in sizes]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([f"k={k}" for k in sizes], counts, color="#2E5EAA")
for i, c in enumerate(counts):
    ax.text(i, c + max(counts) * 0.02, str(c), ha="center", fontweight="bold")
ax.set_ylabel("Number of frequent itemsets")
ax.set_title(f"Frequent itemsets by size (min_support={MIN_SUPPORT})")
fig.tight_layout()
plt.show()

### Support-threshold sensitivity analysis

Lowering the threshold causes candidates to explode combinatorially – the core scalability problem PCY addresses.

In [ ]:
sweep_supports = [2, 3, 4, 5, 7, 10]
sweep_n_items, sweep_n_pairs, sweep_time = [], [], []
for ms in sweep_supports:
    lv, st = run_apriori(baskets, ms)
    sweep_n_items.append(len(lv.get(1, {})))
    sweep_n_pairs.append(len(lv.get(2, {})))
    sweep_time.append(sum(s["time_sec"] for s in st.values()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(sweep_supports, sweep_n_items, "o-", color="#2E5EAA", label="Frequent 1-itemsets")
ax1.plot(sweep_supports, sweep_n_pairs, "o-", color="#E8743B", label="Frequent 2-itemsets")
ax1.set_xlabel("Minimum support threshold (no. of movies)")
ax1.set_ylabel("Number of frequent itemsets")
ax1.set_title("Frequent itemsets vs. support threshold")
ax1.legend()

ax2.plot(sweep_supports, sweep_time, "o-", color="#C0392B")
ax2.set_xlabel("Minimum support threshold (no. of movies)")
ax2.set_ylabel("Total Apriori running time (s)")
ax2.set_title("Running time vs. support threshold")
fig.tight_layout()
plt.show()

## 3. PCY algorithm (Park-Chen-Yu)

**Pass 1**: count singletons + hash every pair into a bucket array.
**Pass 2**: a pair is a candidate only if both items are frequent *and* its bucket was frequent in pass 1.

**No false negatives**: a truly frequent pair always hashes to a bucket with count ≥ min_support, so it is never filtered out.

In [ ]:
def _hash_pair(pair, num_buckets):
    """Deterministic hash (MD5-based; stable across runs unlike Python's hash())."""
    s = ",".join(sorted(pair))
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % num_buckets


def pcy_pass1(baskets, min_support, num_buckets):
    item_counts  = defaultdict(int)
    bucket_counts = defaultdict(int)
    for basket in baskets:
        items = list(basket)
        for item in items:
            item_counts[item] += 1
        for pair in itertools.combinations(sorted(items), 2):
            bucket_counts[_hash_pair(pair, num_buckets)] += 1
    frequent_items   = {i for i, c in item_counts.items()  if c >= min_support}
    frequent_buckets = {b for b, c in bucket_counts.items() if c >= min_support}
    return frequent_items, frequent_buckets


def pcy_pass2(baskets, frequent_items, frequent_buckets, min_support, num_buckets):
    # Count only pairs that pass both filters
    candidate_counts = defaultdict(int)
    for basket in baskets:
        items = sorted(i for i in basket if i in frequent_items)
        for pair in itertools.combinations(items, 2):
            if _hash_pair(pair, num_buckets) in frequent_buckets:
                candidate_counts[frozenset(pair)] += 1
    return {p: c for p, c in candidate_counts.items() if c >= min_support}


def run_pcy_pairs(baskets, min_support, num_buckets=2000):
    t0 = time.perf_counter()
    frequent_items, frequent_buckets = pcy_pass1(baskets, min_support, num_buckets)
    t1 = time.perf_counter()
    frequent_pairs = pcy_pass2(baskets, frequent_items, frequent_buckets, min_support, num_buckets)
    t2 = time.perf_counter()

    # Actual candidate-set size passed to pass 2
    n_pcy_candidates = sum(
        1 for a, b in itertools.combinations(sorted(frequent_items), 2)
        if _hash_pair((a, b), num_buckets) in frequent_buckets
    )
    stats = {
        "n_frequent_items":   len(frequent_items),
        "n_buckets":          num_buckets,
        "n_frequent_buckets": len(frequent_buckets),
        "n_pcy_candidates":   n_pcy_candidates,
        "n_frequent_pairs":   len(frequent_pairs),
        "pass1_time_sec":     t1 - t0,
        "pass2_time_sec":     t2 - t1,
        "total_time_sec":     t2 - t0,
    }
    return frequent_pairs, stats

In [ ]:
pcy_pairs, pcy_stats = run_pcy_pairs(baskets, MIN_SUPPORT, num_buckets=5000)
print(pcy_stats)

# Correctness check: Apriori and PCY must return exactly the same frequent pairs
assert set(levels[2].keys()) == set(pcy_pairs.keys())
print("\nOK: Apriori and PCY find the same set of frequent pairs.")

### Apriori vs PCY: candidate-set size

PCY's main advantage is reducing the **counting table** required in pass 2 – critical when the item catalogue is large and the full pair table would not fit in memory.

In [ ]:
apriori_k2_candidates = apriori_stats[2]["n_candidates"]
pcy_k2_candidates     = pcy_stats["n_pcy_candidates"]
reduction_pct = 100 * (1 - pcy_k2_candidates / apriori_k2_candidates)

print(f"Apriori candidates (all pairs of L1): {apriori_k2_candidates}")
print(f"PCY candidates (after hash filter):   {pcy_k2_candidates}")
print(f"Reduction: {reduction_pct:.1f}%")

fig, ax = plt.subplots(figsize=(5.5, 4))
bars = ax.bar(["Apriori", "PCY"], [apriori_k2_candidates, pcy_k2_candidates],
              color=["#2E5EAA", "#3CB371"])
for b in bars:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, h + apriori_k2_candidates * 0.01,
            str(int(h)), ha="center", fontweight="bold")
ax.set_ylabel("Candidate-set size (pairs)")
ax.set_title(f"Candidate-set reduction: Apriori vs PCY\n"
             f"(min_support={MIN_SUPPORT}, {pcy_stats['n_buckets']} buckets)")
fig.tight_layout()
plt.show()

### Effect of the number of buckets

Too few buckets → many collisions → every bucket becomes frequent → PCY degenerates to Apriori. Too many → collisions drop → candidates approach the true frequent-pair count.

In [ ]:
bucket_sweep = [50, 200, 1000, 5000, 20000, 50000]
bucket_candidates = []
for nb in bucket_sweep:
    _, st = run_pcy_pairs(baskets, MIN_SUPPORT, num_buckets=nb)
    bucket_candidates.append(st["n_pcy_candidates"])

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(bucket_sweep, bucket_candidates, "o-", color="#8E44AD")
ax.axhline(apriori_k2_candidates, color="#2E5EAA", linestyle="--",
           label="Apriori (no filtering)")
ax.axhline(len(levels[2]), color="#3CB371", linestyle="--",
           label="True number of frequent pairs")
ax.set_xscale("log")
ax.set_xlabel("Number of hash buckets (log scale)")
ax.set_ylabel("PCY candidate-set size")
ax.set_title("Effect of bucket count on PCY filtering")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Association rules

Rule form: **I → j** (single-item consequent, per course definition).

- **confidence** = support(I ∪ {j}) / support(I)
- **interest** = confidence − support({j})/N  (positive: I encourages j; negative: I discourages j)

In [ ]:
def generate_association_rules(levels, n_baskets, min_confidence=0.5):
    """Generate rules I -> j (single-item j) from all frequent itemsets of size >= 2."""
    support = {}
    for itemsets in levels.values():
        support.update(itemsets)
    singleton_support = {next(iter(k)): v for k, v in levels.get(1, {}).items()}

    rules = []
    for k, itemsets in levels.items():
        if k < 2:
            continue
        for itemset, itemset_support in itemsets.items():
            for j in itemset:
                antecedent  = itemset - frozenset([j])
                ant_support = support.get(antecedent)
                if not ant_support:
                    continue
                confidence = itemset_support / ant_support
                if confidence < min_confidence:
                    continue
                j_prob   = singleton_support.get(j, 0) / n_baskets
                interest = confidence - j_prob
                rules.append({
                    "antecedent": antecedent, "consequent": j,
                    "support": itemset_support, "confidence": confidence,
                    "interest": interest,
                })
    # Sort by absolute interest (most informative first)
    rules.sort(key=lambda r: abs(r["interest"]), reverse=True)
    return rules

rules = generate_association_rules(levels, N, min_confidence=0.5)
print(f"Rules generated (confidence >= 0.5): {len(rules)}\n")
for r in rules[:10]:
    ant = ", ".join(sorted(r["antecedent"]))
    print(f"  {{{ant}}} -> {r['consequent']}"
          f"   support={r['support']}  conf={r['confidence']:.2f}  interest={r['interest']:+.3f}")

## 5. Insight: the most recurring actor pairs

In [ ]:
top_pairs = sorted(levels[2].items(), key=lambda x: -x[1])[:10]
labels = [" & ".join(sorted(p)) for p, _ in top_pairs]
values = [c for _, c in top_pairs]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(labels[::-1], values[::-1], color="#2E5EAA")
ax.set_xlabel("Number of movies together (support)")
ax.set_title("Top 10 most recurring actor pairs")
fig.tight_layout()
plt.show()

## 6. Conclusions

- With `min_support=3`: **271 frequent actors**, **25 frequent pairs**, **3 frequent triples** (k=4 yields nothing).
- The top pairs/triples correspond to recurring franchise casts (Harry Potter, Lord of the Rings, Star Wars).
- Apriori and PCY find **exactly the same** frequent pairs; PCY reduces the candidate set by >85% with a well-chosen bucket count.
- Lowering the threshold causes combinatorial explosion in candidates – confirming why PCY matters at scale.

**Possible extensions**: larger dataset (not just top 1000), SON algorithm for distributed execution (Spark/MapReduce).